In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append('../../../code/')
import utils
import leakage
import reframed
from collections import defaultdict

In [6]:
model_ez = reframed.load_cbmodel('../../../models/e_coli/momentiJO1366_modified.xml')
model_ez.solver = 'gurobi'

In [7]:
glc_uptake = -7.0435835942413485 # From paczia data

# Estimate metabolite value for all metabolites

In [23]:
metabolites = []
met_to_kegg = {}
for m_id in model_ez.metabolites:
    m = model_ez.metabolites[m_id]
    if m.compartment == 'c':
        kegg_id = m.metadata.get('kegg.compound')
        if kegg_id is not None:
            metabolites.append(m_id)
            met_to_kegg[m_id] = kegg_id

        

In [25]:

shadow_prices = leakage.estimate_shadow_prices_reframed(model_ez, constraints={'R_EX_glc__D_e': (glc_uptake,0)}, metabolites=metabolites)

234/782: M_ca2_c, predicted flux: -0.003256593225447957, secretion reaction: R_EX_ca2_e
252/782: M_cl_c, predicted flux: -0.003256593225447957, secretion reaction: R_EX_cl_e
254/782: M_co2_c, predicted flux: 16.578908719497573, secretion reaction: R_EX_co2_e
352/782: M_fe2_c, predicted flux: -0.01004882685762145, secretion reaction: R_EX_fe2_e
370/782: M_g1p_c, predicted flux: -7.0435835942413485, secretion reaction: R_EX_glc__D_e
376/782: M_g6p_c, predicted flux: -7.0435835942413485, secretion reaction: R_EX_glc__D_e
397/782: M_glc__D_c, predicted flux: -7.0435835942413485, secretion reaction: R_EX_glc__D_e
433/782: M_h_c, predicted flux: 5.748767355435882, secretion reaction: R_EX_h_e
436/782: M_h2o_c, predicted flux: 33.10260872489215, secretion reaction: R_EX_h2o_e
476/782: M_k_c, predicted flux: -0.12212568711908993, secretion reaction: R_EX_k_e
521/782: M_mg2_c, predicted flux: -0.005427655375746595, secretion reaction: R_EX_mg2_e
547/782: M_nh4_c, predicted flux: -6.757711866984

In [28]:
kegg_to_met = defaultdict(list)
for met, kegg_id_list in met_to_kegg.items():
    for kegg_id in kegg_id_list:
        kegg_to_met[kegg_id].append(met)

In [39]:
data = []
for kegg_id, met_list in kegg_to_met.items():
    prices = [shadow_prices[met] for met in met_list]
    avg_price = -np.nanmean(prices)
    data.append([kegg_id, ';'.join(met_list), avg_price])


/var/folders/xf/kl76knj11y72v0_qy4vv7tgh0000gp/T/ipykernel_80080/1639432123.py:4: RuntimeWarning: Mean of empty slice
  avg_price = -np.nanmean(prices)


In [40]:
mv_df = pd.DataFrame(data, columns=['kegg_id', 'metabolite_ids', 'Metabolite value'])

In [46]:
met_to_kegg['M_xylu__L_c']

['C00310', 'C00312']

In [58]:
mv_df.to_csv('../../../data/this_project/6_transporterKO/Z_div/I_metabolite_values_big_screen.csv', index=False)